In [2]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import zscore

In [3]:
df = pd.read_csv('academic_meals_elementary_district.csv')

In [8]:
ses_cols = ['student_low_income_percent', 'student_disabilities_percent']
hei_total_col = ['HEI 2015 Total Score']
hei_components_cols = [
    'HEI 2015 Total Fruits (0-5)', 'HEI 2015 Total Vegetables (0-5)',
    'HEI 2015 Whole Grains (0-10)', 'HEI 2015 Dairy (0-10)',
    'HEI 2015 Total Protein Foods (0-5)', 'HEI 2015 Seafood and Plant Proteins (0-5)',
    'HEI 2015 Fatty Acids (0-10)', 'HEI 2015 Refined Grains (0-10)',
    'HEI 2015 Sodium (0-10)', 'HEI 2015 Added Sugars (0-10)', 'HEI 2015 Saturated Fats (0-10)'
]
targets = ['ELA_Proficiency', 'Math_Proficiency', 'Science_Proficiency']
df_clean = df[ses_cols + hei_total_col + hei_components_cols + targets].dropna()

df_beta = df_clean.apply(zscore)

In [28]:
def check_vif(X):
    vif_df = pd.DataFrame()
    vif_df["Feature"] = X.columns
    vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_df

In [19]:
print("Model 1: HEI Total Score Only")
X1 = sm.add_constant(df_clean[hei_total_col])

for target in targets:
    m1 = sm.OLS(df_clean[target], X1).fit()
    print(f"{target:20} | R²: {m1.rsquared:.4f} | Coef: {m1.params[hei_total_col[0]]:.4f} | p-val: {m1.pvalues[hei_total_col[0]]:.4f}")

Model 1: HEI Total Score Only
ELA_Proficiency      | R²: 0.1267 | Coef: -0.7390 | p-val: 0.0177
Math_Proficiency     | R²: 0.0808 | Coef: -0.5920 | p-val: 0.0614
Science_Proficiency  | R²: 0.0682 | Coef: -0.5101 | p-val: 0.0869


In [20]:
print("Model 2: SES Factors Only")
X2 = sm.add_constant(df_clean[ses_cols])

for target in targets:
    m2 = sm.OLS(df_clean[target], X2).fit()
    print(f"{target:20} | R²: {m2.rsquared:.4f} | Low-Income Coef: {m2.params['student_low_income_percent']: .4f}")

Model 2: SES Factors Only
ELA_Proficiency      | R²: 0.6137 | Low-Income Coef: -0.4919
Math_Proficiency     | R²: 0.6677 | Low-Income Coef: -0.5142
Science_Proficiency  | R²: 0.6042 | Low-Income Coef: -0.4375


In [21]:
print("Model 3: SES + HEI Total Score")

for target in targets:
    X3 = sm.add_constant(df_clean[ses_cols + hei_total_col])
    m3 = sm.OLS(df_clean[target], X3).fit()
    print(f"{target:20} | Adj. R²: {m3.rsquared_adj:.4f} | HEI p-val: {m3.pvalues[hei_total_col[0]]:.4f}")

Model 3: SES + HEI Total Score
ELA_Proficiency      | Adj. R²: 0.6127 | HEI p-val: 0.0974
Math_Proficiency     | Adj. R²: 0.6490 | HEI p-val: 0.4039
Science_Proficiency  | Adj. R²: 0.5863 | HEI p-val: 0.2914


In [31]:
print("Model 4: SES + Components for ELA")
target_ela = 'ELA_Proficiency'
print(f"\n{'='*30} {target_ela} MODEL 4 {'='*30}")

X4_ela = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_ela = sm.OLS(df_clean[target_ela], X4_ela).fit()
print(model4_ela.summary())


Model 4: SES + Components for ELA

============================== ELA_Proficiency MODEL 4 ==============================
                            OLS Regression Results                            
Dep. Variable:        ELA_Proficiency   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.608
Method:                 Least Squares   F-statistic:                     6.130
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           2.08e-05
Time:                        11:58:12   Log-Likelihood:                -162.11
No. Observations:                  44   AIC:                             352.2
Df Residuals:                      30   BIC:                             377.2
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|    

In [30]:
print("\nVIF Check for ELA Model")
print(check_vif(X4_ela))


VIF Check for ELA Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [32]:
X_beta_ela = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_ela = sm.OLS(df_beta[target_ela], X_beta_ela).fit()

print("\n\nTop 5 Beta Estimates")
ela_betas = model_beta_ela.params.drop('const').abs().sort_values(ascending=False)
for i in range(5):
    var = ela_betas.index[i]
    print(f"{i+1}. {var:40} | Beta: {model_beta_ela.params[var]:.4f}")



Top 5 Beta Estimates
1. student_low_income_percent               | Beta: -0.6553
2. HEI 2015 Saturated Fats (0-10)           | Beta: 0.3484
3. HEI 2015 Refined Grains (0-10)           | Beta: -0.2612
4. HEI 2015 Total Protein Foods (0-5)       | Beta: -0.2560
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2516


In [33]:
print("Model 4: SES + Components for Math")
target_math = 'Math_Proficiency'
print(f"\n{'='*30} {target_math} MODEL 4 {'='*30}")

X4_math = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_math = sm.OLS(df_clean[target_math], X4_math).fit()

print(model4_math.summary())

Model 4: SES + Components for Math

============================== Math_Proficiency MODEL 4 ==============================
                            OLS Regression Results                            
Dep. Variable:       Math_Proficiency   R-squared:                       0.773
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     7.857
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           1.74e-06
Time:                        12:04:01   Log-Likelihood:                -158.14
No. Observations:                  44   AIC:                             344.3
Df Residuals:                      30   BIC:                             369.3
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|  

In [34]:
print("\nVIF Check for Math Model")
print(check_vif(X4_math))

X_beta_math = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_math = sm.OLS(df_beta[target_math], X_beta_math).fit()



VIF Check for Math Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [35]:
print("\nTop 5 Beta Estimates")
math_betas = model_beta_math.params.drop('const').abs().sort_values(ascending=False)
for i in range(5):
    var = math_betas.index[i]
    print(f"{i+1}. {var:40} | Beta: {model_beta_math.params[var]:.4f}")


Top 5 Beta Estimates
1. student_low_income_percent               | Beta: -0.7316
2. HEI 2015 Refined Grains (0-10)           | Beta: -0.3114
3. HEI 2015 Saturated Fats (0-10)           | Beta: 0.2386
4. HEI 2015 Whole Grains (0-10)             | Beta: -0.2214
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2113


In [36]:
print("Model 4: SES + Components for Science")
target_sci = 'Science_Proficiency'
print(f"\n{'='*30} {target_sci} MODEL 4 {'='*30}")

X4_sci = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_sci = sm.OLS(df_clean[target_sci], X4_sci).fit()

print(model4_sci.summary())

Model 4: SES + Components for Science

============================== Science_Proficiency MODEL 4 ==============================
                             OLS Regression Results                            
Dep. Variable:     Science_Proficiency   R-squared:                       0.721
Model:                             OLS   Adj. R-squared:                  0.600
Method:                  Least Squares   F-statistic:                     5.969
Date:                 Tue, 24 Mar 2026   Prob (F-statistic):           2.68e-05
Time:                         12:05:16   Log-Likelihood:                -159.85
No. Observations:                   44   AIC:                             347.7
Df Residuals:                       30   BIC:                             372.7
Df Model:                           13                                         
Covariance Type:             nonrobust                                         
                                                coef    std err        

In [37]:
print("\nVIF Check for Science Model")
print(check_vif(X4_sci))


VIF Check for Science Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [38]:
X_beta_sci = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_sci = sm.OLS(df_beta[target_sci], X_beta_sci).fit()

print("\nTop 5 Beta Estimates")
sci_betas = model_beta_sci.params.drop('const').abs().sort_values(ascending=False)
for i in range(5):
    var = sci_betas.index[i]
    print(f"{i+1}. {var:40} | Beta: {model_beta_sci.params[var]:.4f}")


Top 5 Beta Estimates
1. student_low_income_percent               | Beta: -0.6261
2. HEI 2015 Whole Grains (0-10)             | Beta: -0.3241
3. HEI 2015 Saturated Fats (0-10)           | Beta: 0.2899
4. HEI 2015 Total Protein Foods (0-5)       | Beta: -0.2853
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2376


Findings from Beta Estimates

SES Dominance: Low-income status (student_low_income_percent) is the strongest predictor across all subjects, with the most severe impact on Math (Beta: -0.7316).

Saturated Fats as a Primary Driver: Lower intake of saturated fats (higher HEI score) is the top nutritional predictor for both ELA (Beta: 0.3484) and Science (Beta: 0.2899).

Subject-Specific Sensitivity: Math is uniquely sensitive to Refined Grain quality (Beta: -0.3114). Science is uniquely sensitive to Whole Grain quality (Beta: -0.3241).
